### Imports

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
)

from xgboost import XGBClassifier

import mlflow

c:\Users\yash\OneDrive\Desktop\coding\python\SupportTicket-ComplaintClassifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setting MLFlow

In [43]:
mlflow.set_tracking_uri("http://localhost:5000")

print(mlflow.get_tracking_uri())

mlflow.set_experiment("Complaint Text Classification")

http://localhost:5000


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1789749848824, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789749848824, lifecycle_stage='active', name='Complaint Text Classification', tags={}, trace_location=None, workspace='default'>

### Data Preparation

In [14]:
train = pd.read_parquet("../data/train.parquet")
val = pd.read_parquet("../data/val.parquet")
test = pd.read_parquet("../data/test.parquet")

text_col = "clean_complaint_text"

In [15]:
X_train = train[text_col]
y_train = train["Product"]

X_val = val[text_col]
y_val = val["Product"]

X_test = test[text_col]
y_test = test["Product"]

## Logistic Regression Pipeline

In [ ]:
pipeline = Pipeline([
    ("tf-idf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=5
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

### Model Training - Logging

In [7]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Logistic Regression"):
    pipeline.fit(X_train, y_train)

    val_predictions_lr = pipeline.predict(X_val)

    accuracy = accuracy_score(y_val, val_predictions_lr)
    macro_f1 = f1_score(y_val, val_predictions_lr, average="macro")
    weighted_f1 = f1_score(y_val, val_predictions_lr, average="weighted")
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("weighted_f1", weighted_f1)

    cm = confusion_matrix(y_val, val_predictions_lr)
    
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=pipeline.classes_,
        yticklabels=pipeline.classes_
    )
    
    plt.title("Logistic Regression - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    
    plt.tight_layout()

    plt.savefig("lr_confusion_matrix.png")
    plt.close()

    mlflow.log_artifact("lr_confusion_matrix.png")

2026/09/18 22:14:22 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
2026/09/18 22:14:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/18 22:15:06 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
c:\Users\yash\OneDrive\Desktop\coding\python\SupportTicket-ComplaintClassifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/09/18 22:15:56 WARNING mlflow.sklearn: Unrecognized dataset type <class 'panda

🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/1/runs/2fd9e4d1bccd4d05bf4b2c33498ae33a
🧪 View experiment at: http://localhost:5000/#/experiments/1


### Model Report

In [8]:
print(classification_report(y_val, val_predictions_lr))
print(accuracy_score(y_val, val_predictions_lr))

                                                         precision    recall  f1-score   support

                            Checking or savings account       0.72      0.76      0.74       938
                                            Credit card       0.83      0.80      0.81       938
    Credit reporting or other personal consumer reports       0.83      0.92      0.87       937
                                        Debt collection       0.85      0.86      0.85       937
     Money transfer, virtual currency, or money service       0.79      0.79      0.79       937
                                               Mortgage       0.96      0.92      0.94       938
Payday loan, title loan, personal loan, or advance loan       0.78      0.78      0.78       937
                                  Vehicle loan or lease       0.88      0.83      0.85       938

                                               accuracy                           0.83      7500
                            

## XGBoost Classifier Pipeline

In [ ]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

import os, pickle

os.makedirs("../models", exist_ok=True)

with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

xgb_pipeline = Pipeline([
    ("tf-idf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=5
    )),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        tree_method="hist",
        n_jobs=8,
        colsample_bytree=0.3,
        colsample_bylevel=0.5,
        max_bin=128
    ))
])

### Model Training - Logging

In [10]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="XGBoost - TF-IDF"):

    xgb_pipeline.fit(X_train, y_train_encoded)

    val_predictions_xgb_encoded = xgb_pipeline.predict(X_val)

    val_predictions_xgb = label_encoder.inverse_transform(
        val_predictions_xgb_encoded.astype(int)
    )

    accuracy = accuracy_score(y_val, val_predictions_xgb)
    macro_f1 = f1_score(y_val, val_predictions_xgb, average="macro")
    weighted_f1 = f1_score(y_val, val_predictions_xgb, average="weighted")
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)
    mlflow.log_metric("weighted_f1", weighted_f1)
    
    cm = confusion_matrix(y_val, val_predictions_xgb)
    
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    
    plt.title("XGBoost - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    
    plt.tight_layout()

    plt.savefig("xgb_confusion_matrix.png")
    plt.close()
    
    mlflow.log_artifact("xgb_confusion_matrix.png")

2026/09/18 22:15:57 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
2026/09/18 22:42:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/18 22:42:23 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
c:\Users\yash\OneDrive\Desktop\coding\python\SupportTicket-ComplaintClassifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/09/18 22:43:22 WARNING mlflow.sklearn: Unrecognized dataset type <class 'panda

🏃 View run XGBoost - TF-IDF at: http://localhost:5000/#/experiments/1/runs/30d81788dbce403e8e74e2c865023f2d
🧪 View experiment at: http://localhost:5000/#/experiments/1


### Model Report

In [11]:
print(classification_report(y_val, val_predictions_xgb))
print(accuracy_score(y_val, val_predictions_xgb))

                                                         precision    recall  f1-score   support

                            Checking or savings account       0.72      0.73      0.72       938
                                            Credit card       0.84      0.81      0.83       938
    Credit reporting or other personal consumer reports       0.85      0.93      0.89       937
                                        Debt collection       0.86      0.86      0.86       937
     Money transfer, virtual currency, or money service       0.78      0.79      0.78       937
                                               Mortgage       0.96      0.92      0.94       938
Payday loan, title loan, personal loan, or advance loan       0.78      0.77      0.77       937
                                  Vehicle loan or lease       0.89      0.85      0.87       938

                                               accuracy                           0.83      7500
                            